# Name Trend Forecasting

This notebook explores historical baby-name popularity and demonstrates forecasting with linear regression and ARIMA. Optional sections can be extended with Prophet and changepoint detection.

**Author:** Muhammad Kamil Shah

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA


## Load the dataset

Provide a CSV containing at least `Year`, `Name`, and `Count` columns. The sample path below can be changed to match your local file.

In [ ]:
DATA_PATH = '../data/name_trends.csv'
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
required = {'Year', 'Name', 'Count'}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

df = df.dropna(subset=['Year', 'Name', 'Count']).copy()
df['Year'] = df['Year'].astype(int)
df['Count'] = pd.to_numeric(df['Count'], errors='coerce')
df = df.dropna(subset=['Count'])


## Explore one name

In [ ]:
selected_name = df.groupby('Name')['Count'].sum().idxmax()
series = (df[df['Name'] == selected_name]
          .groupby('Year', as_index=False)['Count'].sum()
          .sort_values('Year'))

plt.figure(figsize=(11, 5))
plt.plot(series['Year'], series['Count'])
plt.title(f'Historical popularity of {selected_name}')
plt.xlabel('Year')
plt.ylabel('Count')
plt.grid(alpha=0.3)
plt.show()

## Linear-regression baseline

In [ ]:
split = max(5, int(len(series) * 0.8))
train, test = series.iloc[:split], series.iloc[split:]
model = LinearRegression().fit(train[['Year']], train['Count'])
pred = model.predict(test[['Year']])
print('MAE:', mean_absolute_error(test['Count'], pred))
print('RMSE:', mean_squared_error(test['Count'], pred) ** 0.5)

## ARIMA forecast

In [ ]:
arima = ARIMA(series['Count'], order=(2, 1, 2)).fit()
future_steps = 10
forecast = arima.forecast(steps=future_steps)
future_years = np.arange(series['Year'].max() + 1, series['Year'].max() + future_steps + 1)

plt.figure(figsize=(11, 5))
plt.plot(series['Year'], series['Count'], label='Historical')
plt.plot(future_years, forecast, marker='o', label='ARIMA forecast')
plt.title(f'Forecast for {selected_name}')
plt.xlabel('Year')
plt.ylabel('Count')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Conclusion

The notebook establishes a reproducible baseline for name-popularity forecasting. Future work should compare multiple names, tune forecasting parameters, add Prophet, and evaluate rolling-origin forecasts.